# 74 — End-to-end: SASRec recall + wRRF union -> LGBM rerank -> nDCG, + Blind-A submission

One notebook for the whole improved pipeline.

Stage 1 (recall): content-fused SASRec as a 4th channel in wrrf_union_v1. Reports recall@{20,100} for union vs union+SASRec (the G1 result; +0.0586 @100 on full dev).

Stage 2 (rerank train): build LGBM LambdaRank features WITH the SASRec channel (sasrec_rank_inv feature) and train two models — with and without that feature — reporting held-out val nDCG@20 (the G2 number + its control).

Stage 3 (end-to-end DEV nDCG): union+SASRec recall@100 -> LGBM rerank -> nDCG@20 on dev (we have golds here). Compares recall-only vs LGBM(no sasrec feat) vs LGBM(+sasrec feat).

Stage 4 (Blind-A submission, separate logic at the end): run the full pipeline (config 191 = union+SASRec -> LGBM(+sasrec) -> v5-kto responder) over the 80 Blind-A queries and package prediction.json. NOTE: Blind-A has no public golds — nDCG there is scored by the CodaBench server, not locally. Stage 3 is the local nDCG signal.

Run order: cells top to bottom. Stages 1-3 are fast; Stage 4 is the long responder run (~35-65 min) — run it only when Stage 3 confirms the lift.

In [ ]:
# 1) Setup. Disable JAX GPU preallocation BEFORE any import pulls JAX in
# (datasets/transformers import JAX transitively; it grabs ~75% VRAM on first use).
import os
os.environ.setdefault('XLA_PYTHON_CLIENT_PREALLOCATE', 'false')
os.environ.setdefault('TF_FORCE_GPU_ALLOW_GROWTH', 'true')
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '3')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
from google.colab import userdata, drive
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
drive.mount('/content/drive', force_remount=False)

BRANCH = 'recall-union-lgbm'
!rm -rf /content/recsys2026
!git clone -b {BRANCH} https://github.com/orrimoch/recsys2026-lora-tutorial.git /content/recsys2026
%cd /content/recsys2026

# Symlink the persistent caches from Drive. retrieval_v2 holds sasrec/, lgbm/,
# ctx_cache/; dense holds the Qwen query/catalog cache for dense_metadata_qwen3.
DRIVE_BASE = '/content/drive/MyDrive'
LOCAL_BASE = '/content/recsys2026/experiments/cache'
os.makedirs(LOCAL_BASE, exist_ok=True)
for name, drive_subdir in [
    ('retrieval_v2', 'recsys2026_retrieval_v2_cache'),
    ('dense', 'recsys2026_dense_cache'),
]:
    src = f'{DRIVE_BASE}/{drive_subdir}'
    dst = f'{LOCAL_BASE}/{name}'
    os.makedirs(src, exist_ok=True)
    if os.path.islink(dst): os.unlink(dst)
    elif os.path.exists(dst):
        import shutil; shutil.rmtree(dst)
    os.symlink(src, dst)

# Deps: retrieval stack + lightgbm (Stages 1-3) + responder/inference (Stage 4).
!pip install -q --upgrade 'transformers>=4.40' 'accelerate>=0.30' 'peft>=0.11'     'datasets' 'pandas<3.0' 'tqdm' 'huggingface_hub' 'sentence-transformers>=3.0'     'FlagEmbedding>=1.3' 'bm25s' 'lightgbm' 'scikit-learn'     'omegaconf' 'pyyaml' 'trl>=0.12.0' 'torchao>=0.17'


## Stage 1 — recall: content-fused SASRec as the 4th union channel

In [ ]:
# 3) Ensure the content-fused SASRec checkpoint exists (sasrec_v1). Trains it
# only if missing (it persists on the Drive cache across runtimes).
import os, sys
sys.path.insert(0, '/content/recsys2026/music-crs-baselines')
CACHE_DIR = '/content/recsys2026/experiments/cache'
ITEM_DB = 'talkpl-ai/TalkPlayData-Challenge-Track-Metadata'
CORPUS = ['track_name', 'artist_name', 'album_name']
SASREC_CKPT = f'{CACHE_DIR}/retrieval_v2/sasrec/sasrec_v1/sasrec.pt'
if os.path.exists(SASREC_CKPT):
    print('[sasrec] checkpoint present, skipping train:', SASREC_CKPT)
else:
    print('[sasrec] training content-fused SASRec (sasrec_v1, ~10 epochs)...')
    !cd /content/recsys2026 && python -u scripts/train_sasrec.py \
        --cache-dir {CACHE_DIR} --out sasrec_v1 --epochs 10


In [ ]:
# 4) Build the FULL dev eval set + report recall@{20,100} for union vs union+SASRec.
# Defines the shared dev variables reused by Stage 3.
import numpy as np
import pandas as pd
from datasets import load_dataset
from mcrs.db_item.music_catalog import MusicCatalogDB
from mcrs.retrieval_modules import load_retrieval_module
from mcrs.retrieval_modules.sasrec_model import build_user_dialog

item_db = MusicCatalogDB(ITEM_DB, ['all_tracks'], CORPUS)
dev = load_dataset('talkpl-ai/TalkPlayData-Challenge-Dataset', split='test')

queries, golds, user_ids, played, user_dialogs = [], [], [], [], []
goal_categories, goal_specificities, turn_numbers = [], [], []
for sess in dev:
    df = pd.DataFrame(sess['conversations'])
    goal = sess.get('conversation_goal') or {}
    for _, music in df[df['role'] == 'music'].iterrows():
        tn = int(music['turn_number'])
        prior = df[(df['turn_number'] < tn) |
                   ((df['turn_number'] == tn) & (df['role'] == 'user'))]
        lines = []
        for _, t in prior.iterrows():
            role = 'assistant' if t['role'] == 'music' else t['role']
            content = item_db.id_to_metadata(t['content']) if t['role'] == 'music' else t['content']
            lines.append(f'{role}: {content}')
        queries.append(chr(10).join(lines))
        user_dialogs.append(build_user_dialog(prior.to_dict('records')))
        golds.append(music['content'])
        user_ids.append(sess.get('user_id'))
        played.append(list(df[(df['role'] == 'music') & (df['turn_number'] < tn)]['content']))
        goal_categories.append(goal.get('category'))
        goal_specificities.append(goal.get('specificity'))
        turn_numbers.append(tn)
ctx = [{'history_tids': p, 'user_dialog': ud} for p, ud in zip(played, user_dialogs)]
print('[dev] built', len(queries), 'turns')

def recall_at(cands, k):
    return float(np.mean([1.0 if g in c[:k] else 0.0 for c, g in zip(cands, golds)]))

base = load_retrieval_module('wrrf_union_v1', ITEM_DB, ['all_tracks'], CORPUS,
                             CACHE_DIR, extra_config={})
sas = load_retrieval_module('wrrf_union_v1', ITEM_DB, ['all_tracks'], CORPUS,
                            CACHE_DIR, extra_config={'use_sasrec': True, 'w_sasrec': 1.0})
cb = base.batch_text_to_item_retrieval(queries, topk=100, user_ids=user_ids, batch_context=ctx)
cs = sas.batch_text_to_item_retrieval(queries, topk=100, user_ids=user_ids, batch_context=ctx)
print('=== Stage 1 recall (FULL dev, n=' + str(len(golds)) + ') ===')
print('  union (3-chan) : @20=' + str(round(recall_at(cb, 20), 4)) + ' @100=' + str(round(recall_at(cb, 100), 4)))
print('  union + SASRec : @20=' + str(round(recall_at(cs, 20), 4)) + ' @100=' + str(round(recall_at(cs, 100), 4)))
print('  delta @100     :', round(recall_at(cs, 100) - recall_at(cb, 100), 4))


## Stage 2 — LGBM rerank: build features (with SASRec) + train (with vs without the SASRec feature)

In [ ]:
# 6) Build LGBM LambdaRank features from train sessions, WITH the SASRec channel
# (emits the real sasrec_rank_inv column). Two seeds -> train + val splits.
import os
LGBM_DIR = f'{CACHE_DIR}/retrieval_v2/lgbm'
os.makedirs(LGBM_DIR, exist_ok=True)
TRAIN_FEAT = f'{LGBM_DIR}/lgbm_train_sasrec.parquet'
VAL_FEAT   = f'{LGBM_DIR}/lgbm_val_sasrec.parquet'
if os.path.exists(TRAIN_FEAT) and os.path.exists(VAL_FEAT):
    print('[lgbm] feature parquets present, skipping build')
else:
    !cd /content/recsys2026 && python -u scripts/build_lgbm_features.py \
        --n-sessions 2000 --topk 100 --seed 42 \
        --use-sasrec --w-sasrec 1.0 --sasrec-model-dir sasrec_v1 \
        --cache-dir {CACHE_DIR} --out {TRAIN_FEAT}
    !cd /content/recsys2026 && python -u scripts/build_lgbm_features.py \
        --n-sessions 400 --topk 100 --seed 7 \
        --use-sasrec --w-sasrec 1.0 --sasrec-model-dir sasrec_v1 \
        --cache-dir {CACHE_DIR} --out {VAL_FEAT}


In [ ]:
# 7) Train THREE LGBM models on the SAME feature build to isolate the in-sample
# model-derived feature leak (see project_sasrec_lgbm_feature_leak memory):
#   lgbm_sasrec_v1   : all columns (sasrec_rank_inv + cfbpr_score present)
#   lgbm_nosasrec_v1 : sasrec_rank_inv dropped (cfbpr_score still present)
#   lgbm_clean_v1    : BOTH sasrec_rank_inv AND cfbpr_score dropped  <-- LEAK TEST
# train_lgbm_ranker auto-selects every non-id column as a feature, so dropping a
# column is the clean one-axis control. Cheap (CPU, no-GPU) confirmatory test:
# if lgbm_clean_v1 matches/beats recall-only on dev (cell 9) while the others lose,
# the leak is confirmed as the reranker's whole problem.
import os, json
import pandas as pd
LGBM_DIR = f'{CACHE_DIR}/retrieval_v2/lgbm'
TRAIN_FEAT = f'{LGBM_DIR}/lgbm_train_sasrec.parquet'
VAL_FEAT   = f'{LGBM_DIR}/lgbm_val_sasrec.parquet'
TRAIN_NS   = f'{LGBM_DIR}/lgbm_train_nosasrec.parquet'
VAL_NS     = f'{LGBM_DIR}/lgbm_val_nosasrec.parquet'
TRAIN_CL   = f'{LGBM_DIR}/lgbm_train_clean.parquet'
VAL_CL     = f'{LGBM_DIR}/lgbm_val_clean.parquet'
# nosasrec control: drop only sasrec_rank_inv
for src, dst in [(TRAIN_FEAT, TRAIN_NS), (VAL_FEAT, VAL_NS)]:
    df = pd.read_parquet(src)
    df.drop(columns=[c for c in ['sasrec_rank_inv'] if c in df.columns]).to_parquet(dst, index=False)
print('[lgbm] built no-sasrec control parquets')
# clean (leak test): drop BOTH model-derived leaked features
LEAKED = ['sasrec_rank_inv', 'cfbpr_score']
for src, dst in [(TRAIN_FEAT, TRAIN_CL), (VAL_FEAT, VAL_CL)]:
    df = pd.read_parquet(src)
    df.drop(columns=[c for c in LEAKED if c in df.columns]).to_parquet(dst, index=False)
print('[lgbm] built clean (no-leak) parquets \u2014 dropped', LEAKED)

!cd /content/recsys2026 && python -u scripts/train_lgbm_ranker.py \
    --train-features {TRAIN_FEAT} --val-features {VAL_FEAT} \
    --output-dir {LGBM_DIR}/lgbm_sasrec_v1
!cd /content/recsys2026 && python -u scripts/train_lgbm_ranker.py \
    --train-features {TRAIN_NS} --val-features {VAL_NS} \
    --output-dir {LGBM_DIR}/lgbm_nosasrec_v1
!cd /content/recsys2026 && python -u scripts/train_lgbm_ranker.py \
    --train-features {TRAIN_CL} --val-features {VAL_CL} \
    --output-dir {LGBM_DIR}/lgbm_clean_v1

print('\n=== Stage 2 held-out val nDCG@20 (LGBM internal \u2014 leak-inflated, see cell 9 for honest dev) ===')
for name in ['lgbm_clean_v1', 'lgbm_nosasrec_v1', 'lgbm_sasrec_v1']:
    meta = json.load(open(f'{LGBM_DIR}/{name}/metadata.json'))
    print('  ' + name + ': val_ndcg@20=' + str(round(meta['best_val_ndcg20'], 4)) +
        '  (' + str(len(meta['features'])) + ' features)')


## Stage 3 — end-to-end DEV nDCG@20: recall -> rerank

In [ ]:
# 9) End-to-end on dev: union+SASRec recall@100 -> LGBM rerank top-20 -> nDCG@20.
# All LGBM models rerank the SAME union+SASRec pool, so this isolates the rerank
# feature set. The SASRec per-candidate rank is fed exactly as crs_baseline does in
# prod; LGBM_RERANKER only uses features listed in its own metadata, so the clean
# model harmlessly ignores efpc's sasrec_rank and skips cfbpr_score.
# LEAK TEST READING: recall-only is the bar (0.1473 in the run that found the leak).
#   - lgbm_clean_v1 (no leaked feats) >= recall-only  => leak WAS the problem; this
#     leak-free reranker is shippable with NO GPU. OOF only needed to ADD sasrec
#     value on top.
#   - lgbm_clean_v1 still < recall-only               => leak isn't the whole story;
#     OOF would not have helped alone. Investigate label sparsity / wrrf_rank
#     contamination / train-set size next.
import math
from mcrs.rerankers.lgbm_rerank import LGBM_RERANKER
from mcrs.retrieval_modules.rrf import RRF_MODEL

per_sub, labels = sas.batch_per_sub_rankings(queries, user_ids=user_ids, batch_context=ctx)
weights = [s['weight'] for s in sas.subs]
fused100 = RRF_MODEL.fuse_per_sub(per_sub, weights, sas.k, 100)
sidx = labels.index('sasrec_seq')
efpc = []
for qi, cands in enumerate(fused100):
    rm = {tid: r + 1 for r, tid in enumerate(per_sub[sidx][qi])}
    efpc.append([{'sasrec_rank': rm.get(tid, 10000)} for tid in cands])
esi = [{'played_tids': played[i], 'turn_number': turn_numbers[i],
        'prior_track_count': len(played[i])} for i in range(len(queries))]

def ndcg20(ranked):
    s = 0.0
    for r, g in zip(ranked, golds):
        for pos, tid in enumerate(r[:20]):
            if tid == g:
                s += 1.0 / math.log2(pos + 2)
                break
    return s / len(golds)

print('=== Stage 3 end-to-end DEV (n=' + str(len(golds)) + ') ===')
print('  recall@100 pool ceiling      :', round(recall_at(fused100, 100), 4))
recall_only = ndcg20([r[:20] for r in fused100])
print('  nDCG@20 recall-only (no rerank):', round(recall_only, 4), '  <-- bar to beat')
for name, sub in [('LGBM clean (no leaked feats)', 'lgbm_clean_v1'),
                  ('LGBM (no sasrec feat)      ', 'lgbm_nosasrec_v1'),
                  ('LGBM (+ sasrec feat)       ', 'lgbm_sasrec_v1')]:
    rr = LGBM_RERANKER(ITEM_DB, ['all_tracks'], CORPUS, CACHE_DIR,
                       model_path=f'{CACHE_DIR}/retrieval_v2/lgbm/{sub}')
    reranked = rr.rerank(queries, fused100, topk=20, user_ids=user_ids,
                         goal_categories=goal_categories,
                         goal_specificities=goal_specificities,
                         user_profiles_raw=[None] * len(queries),
                         extra_features_per_candidate=efpc, extra_session_info=esi)
    score = ndcg20(reranked)
    flag = '  BEATS recall-only' if score >= recall_only else ''
    print('  nDCG@20 ' + name + ' :', round(score, 4), flag)


## Stage 4 — Blind-A submission (separate logic)

Runs the full pipeline via config 191 (union+SASRec -> LGBM(+sasrec) -> v5-kto responder) over the 80 Blind-A queries and packages prediction.json for CodaBench. This is the LONG cell (~35-65 min). Blind-A nDCG is server-scored — there are no local golds; use Stage 3 as the local signal before submitting.

Requires Stage 2 to have written lgbm_sasrec_v1 to the Drive cache (config 191 points at it).

In [ ]:
# 11) Blind-A inference -> prediction.json -> zip for CodaBench.
import os
TID = '191-union-sasrec-lgbm-v5kto-blindA'
PRED_PATH = f'/content/recsys2026/music-crs-baselines/exp/inference/blindset_A/{TID}.json'
# config 191 reranker_model_path must resolve to the lgbm_sasrec_v1 dir built in Stage 2.
assert os.path.exists(f'{CACHE_DIR}/retrieval_v2/lgbm/lgbm_sasrec_v1/booster.txt'), \
    'Train Stage 2 first — lgbm_sasrec_v1 booster missing.'

%cd /content/recsys2026/music-crs-baselines
!python run_inference_blindset.py --tid {TID} --batch_size 8 2>&1 | tail -60
%cd /content/recsys2026

import json
preds = json.load(open(PRED_PATH))
n = len(preds) if isinstance(preds, list) else len(preds)
print('[blindA] prediction entries:', n)
assert n == 80, f'expected 80 Blind-A entries, got {n} — DO NOT submit'
print('[blindA] sample keys:', list(preds[0].keys()))

# Optional strict precheck (catalog membership + schema):
#   !python scripts/precheck_prediction.py {PRED_PATH}

import zipfile, datetime
zip_path = f'/content/drive/MyDrive/recsys2026_submissions/{datetime.date.today().isoformat()}-{TID}.zip'
os.makedirs(os.path.dirname(zip_path), exist_ok=True)
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    z.write(PRED_PATH, arcname='prediction.json')   # MUST be at zip root for CodaBench
print('[blindA] submission zip ready:', zip_path)
print('[blindA] upload to https://www.codabench.org/competitions/ and append scores via scripts/blind_a_score_tracker.py')
